# Adding Pre-K analysis and using Crunchbase categories

In [2]:
import pandas as pd

from discovery_child_development import PROJECT_DIR
from discovery_child_development.getters import crunchbase

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'


2024-05-24 17:36:55,698 - botocore.credentials - INFO - Found credentials in environment variables.
2024-05-24 17:36:57,352 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load in data

In [3]:
# Crunchbase org big table
organizations_df = crunchbase.get_cb_from_s3(table='organizations')

In [4]:
# Relevant Crunchbase companies
cb_df = pd.read_csv(ENRICHED_DATA_DIR / 'crunchbase_texts_relevant.csv')

In [5]:
# Pre-K project to project categories
taxonomy_iss = pd.read_csv(PROJECT_DIR / 'inputs/data/taxonomy_iss.csv')
prek_to_iss_categories_dict = dict(zip(taxonomy_iss['subtheme'], taxonomy_iss['iss_category']))

# Crunchbase to project categories
cb_to_iss_categories = (
    pd.read_csv(PROJECT_DIR / 'inputs/data/cb_to_iss_categories.csv')
    .query("topic != '-'")
)
cb_to_iss_categories_dict = dict(zip(cb_to_iss_categories['category'], cb_to_iss_categories['topic']))

## Prepare pre-K data

In [6]:
# Match Pre-K analysis to ISS categories

prek_df_matched = (
    # Get Pre-K data
    pd.read_csv(PROJECT_DIR / 'inputs/data/companies_2023_06_23.csv')
    # Map Pre-K to ISS categories
    .assign(iss_category= lambda df: df['subtheme'].map(prek_to_iss_categories_dict))
    # Create a list of ISS categories
    .groupby("cb_id")
    .agg({"iss_category": list})
    .reset_index()
    # Remove nulls
    .assign(iss_category_list = lambda df: df['iss_category'].apply(lambda x: [] if pd.isna(x[0]) else x))
    .drop(columns=['iss_category'])
)

## Merge with newly classified data

In [7]:
cb_df_labelled = (
    # Load in data classified with the new classifiers
    pd.read_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_crunchbase_filtered.csv')
    # Convert comma separated to python list
    .assign(topics_list = lambda df: df.topics.fillna("").apply(lambda x: [t.strip() for t in x.split(",")]))
    # Merge with Pre-K data
    .merge(prek_df_matched, left_on="id", right_on="cb_id", how="outer")
    # Convert NaN to empty list for both topic_list and iss_category_list columns
    .assign(topics_list = lambda df: df['topics_list'].fillna("").apply(lambda x: x if isinstance(x, list) else []))
    .assign(iss_category_list = lambda df: df['iss_category_list'].fillna("").apply(lambda x: x if isinstance(x, list) else []))
    .assign(topics_list_combined = lambda df: (df['topics_list'] + df['iss_category_list']).apply(lambda x: list(set(x)) if type(x) == list else x))
    # Combine id and cb_id columns
    .assign(id = lambda df: df['id'].combine_first(df['cb_id']))
    # Clean up
    .drop(columns=['cb_id', 'topics_list', 'iss_category_list', 'topics'])
    .rename(columns={"topics_list_combined": "topics"})
)

## Merge with Crunchbase category data

In [257]:
# Get all categories
categories_list = (
    organizations_df
    .category_list
    .dropna()
    .apply(lambda x: [s.strip() for s in x.split(",")])
    .to_list()
)
categories_list = sorted(list(set([item for sublist in categories_list for item in sublist])))
pd.DataFrame(categories_list, columns=['category']).to_csv(ENRICHED_DATA_DIR / 'cb_categories_list.csv', index=False)


In [258]:
# Relevant companies
relevant_orgs_df = (
    organizations_df
    .query("id in @cb_df_labelled.id.to_list()")
)

In [259]:
# Relevant crunchbase categories
cb_df_categories = (
    relevant_orgs_df
    # Convert comma separated to python list
    .assign(categories = lambda df: df.category_list.apply(lambda x: [s.strip() for s in x.split(",")] if isinstance(x, str) else []))
    .explode("categories")
    # Map Crunchbase to ISS categories
    .assign(topics = lambda df: df.categories.apply(lambda x: cb_to_iss_categories_dict[x] if x in cb_to_iss_categories_dict else None))
    .groupby("id")
    .agg({"topics": lambda x: list(set(list(x)))})
    .reset_index()
    # Remove empty topics
    .assign(topics = lambda df: df.topics.apply(lambda x: [t for t in x if t is not None]))
)

In [298]:
# Merge both labels and Crunchbase categories
cb_df_labelled_final = (
    cb_df_labelled
    .merge(cb_df_categories, on="id", how="left", suffixes=("", "_cb"))
    .assign(topics_list_combined = lambda df: (df['topics'] + df['topics_cb']).apply(lambda x: list(set(x)) if type(x) == list else x))
    .drop(columns=['topics', 'topics_cb'])
    .rename(columns={"topics_list_combined": "topics"})
    # convert list to comma separated list
    .dropna(subset=['topics'])
    .assign(topics = lambda df: df['topics'].apply(lambda x: ", ".join([t for t in x if ((type(t) is str) and (len(t)> 0))]) if type(x) == list else x))
    .merge(
        organizations_df[['id', 'name', 'cb_url', 'total_funding_usd', 'num_funding_rounds', 'last_funding_on', 'short_description']],
        on="id",
        how="left"
    )
)

In [1]:
organizations_df.columns

NameError: name 'organizations_df' is not defined

In [299]:
cb_df_labelled_final.head(3)

,id,text,topics,name,cb_url,total_funding_usd,num_funding_rounds,last_funding_on,short_description
0,b55fd74c-263e-3140-4fb0-191182984fc3,Club Penguin is an online virtual island where...,games,Club Penguin,https://www.crunchbase.com/organization/clubpe...,NaN,NaN,None,Club Penguin is an online virtual island where...
1,b2940b68-3e4a-b6bc-6764-3d18228f1dda,Sampa is a web platform for families and frien...,social_media,Sampa,https://www.crunchbase.com/organization/sampa,1310000.0,2.0,2008-04-02,Sampa is a web platform for families and frien...
2,e6b40090-6a1d-d506-0bde-135ce10c1589,"BabyCenter, a subsidiary of Johnson & Johnson,...","mobile, parenting2, prenatal, health",BabyCenter,https://www.crunchbase.com/organization/babyce...,10000000.0,2.0,2002-01-02,"BabyCenter, a subsidiary of Johnson & Johnson,..."


In [300]:
cb_df_labelled_final.to_csv(ENRICHED_DATA_DIR / 'crunchbase_combined_labels.csv', index=False)


### Manual checks

In [302]:
checked_df = pd.read_csv(ENRICHED_DATA_DIR / 'crunchbase_combined_labels_checked.csv')


In [304]:
checked_df.head(1)

,id,text,topics,name,cb_url,total_funding_usd,num_funding_rounds,last_funding_on,short_description
0,397845f1-40de-d4f2-8ce8-c0f44161a708,Toys “R” Us is a toy and baby products retaile...,games,Toys R Us Iberia,https://www.crunchbase.com/organization/toys-r-us,3.157000e+09,2.0,2017-09-27,Toys “R” Us is a toy and baby products retaile...
